# eBiblio Book Loan Automation (Google Colab)

This notebook prepares and runs `book-loan.robot` in a Google Colab runtime. Run the cells from top to bottom. You will upload the Robot suite and your private `credentials.resource` file; the notebook does not store credentials in its source.

> **Live action warning:** The live run logs in to eBiblio and may place a reservation or complete a real loan. It can wait up to four hours for today's newspaper. The live run is disabled by default; review the workflow and enable the explicit confirmation only when you intend to proceed. Keep credentials and generated reports private.

## 1. Upload the Robot suite and credentials

Upload both `book-loan.robot` and your own `credentials.resource` when prompted. The credentials file must define non-empty `${USUARIO}` and `${PASSWORD}` variables. The cell writes the files into the temporary Colab runtime and does not display their contents.

In [4]:
from pathlib import Path
import os
import re

from google.colab import files

WORK_DIR = Path("/content/book-loan")
WORK_DIR.mkdir(parents=True, exist_ok=True)

uploaded = files.upload()

# Buscamos archivos que comiencen con el nombre requerido en caso de que Colab les haya añadido un número (ej. 'book-loan (2).robot')
resolved_files = {}
for req in ["book-loan.robot", "credentials.resource"]:
    # Buscar coincidencias en las claves subidas
    stem = Path(req).stem
    suffix = Path(req).suffix
    match = None
    for key in uploaded.keys():
        if key.startswith(stem) and key.endswith(suffix):
            match = key
            break
    if match:
        resolved_files[req] = match

required_files = {"book-loan.robot", "credentials.resource"}
missing_files = required_files.difference(resolved_files.keys())
if missing_files:
    raise ValueError(
        "Upload both required files. Missing: " + ", ".join(sorted(missing_files))
    )

for filename in required_files:
    destination = WORK_DIR / filename
    uploaded_key = resolved_files[filename]
    destination.write_bytes(uploaded[uploaded_key])
    if filename == "credentials.resource":
        os.chmod(destination, 0o600)

robot_source = (WORK_DIR / "book-loan.robot").read_text(encoding="utf-8")
if not re.search(r"(?mi)^\s*Resource\s+credentials\.resource\s*$", robot_source):
    raise ValueError("book-loan.robot must import credentials.resource")

credentials = (WORK_DIR / "credentials.resource").read_text(encoding="utf-8")
for variable in ("USUARIO", "PASSWORD"):
    if not re.search(rf"(?m)^\$\{{{variable}\}}\s+\S", credentials):
        raise ValueError(
            f"credentials.resource must define a non-empty ${{{variable}}} variable"
        )

print("Both input files are present and the credential variables are defined.")

Saving book-loan.robot to book-loan (3).robot
Saving credentials.resource to credentials (2).resource
Both input files are present and the credential variables are defined.


## 2. Check and install the pinned Python dependencies

The project pins Robot Framework and Browser Library in `requirements.txt`. This cell checks the runtime first and installs only missing or mismatched versions.

In [ ]:
import importlib.metadata
import sys
from IPython import get_ipython

required_versions = {
    "robotframework": "7.5",
    "robotframework-browser": "20.6.0",
}
packages_to_install = []
for package, required_version in required_versions.items():
    try:
        installed_version = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        installed_version = None
    print(f"{package}: {installed_version or 'not installed'} (required {required_version})")
    if installed_version != required_version:
        packages_to_install.append(f"{package}=={required_version}")

if packages_to_install:
    get_ipython().run_line_magic("pip", "install " + " ".join(packages_to_install))
else:
    print("Pinned versions are already installed.")

for package, required_version in required_versions.items():
    installed_version = importlib.metadata.version(package)
    if installed_version != required_version:
        raise RuntimeError(
            f"{package} {installed_version} is installed, but {required_version} is required."
        )
print("Python dependencies are ready.")

## 3. Prepare Node.js for Browser Library

Browser Library uses Node.js. Google Colab normally includes it; install the system packages only if either `node` or `npm` is missing.

In [ ]:
import shutil
import subprocess
import sys

if not sys.platform.startswith("linux"):
    raise RuntimeError("Use a Linux-based Google Colab runtime for this notebook.")

if shutil.which("node") is None or shutil.which("npm") is None:
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", "nodejs", "npm"], check=True)

for executable in ("node", "npm"):
    version = subprocess.run(
        [executable, "--version"], check=True, capture_output=True, text=True
    ).stdout.strip()
    print(f"{executable}: {version}")

## 4. Install Chromium for Browser Library

This initializes the Chromium browser that the Robot suite will use in headless mode.

In [ ]:
subprocess.run(
    ["rfbrowser", "init", "chromium"],
    cwd=WORK_DIR,
    check=True,
)
print("Chromium is ready for the Robot Framework Browser Library.")

## 5. Validate without logging in

The dry run checks suite parsing, resource imports, and keyword resolution. It does not open a browser or submit a loan/reservation.

In [ ]:
dry_run_result = subprocess.run(
    [
        sys.executable,
        "-m",
        "robot",
        "--dryrun",
        "--output",
        "NONE",
        "--log",
        "NONE",
        "--report",
        "NONE",
        "--variable",
        "HEADLESS:True",
        str(WORK_DIR / "book-loan.robot"),
    ],
    cwd=WORK_DIR,
    check=False,
)
if dry_run_result.returncode != 0:
    raise RuntimeError(
        f"Robot Framework dry run failed with exit code {dry_run_result.returncode}. "
        "Resolve the reported issue before enabling the live run."
    )
print("Dry run passed; no live browser actions were performed.")

## 6. Optional live loan/reservation attempt

Only change `RUN_LIVE_BOOK_LOAN` to `True` after reviewing this Robot workflow and confirming that you want it to log in and possibly complete a real loan or reservation. Then run this cell. The suite may wait up to four hours for today's edition.

In [ ]:
RUN_LIVE_BOOK_LOAN = False

if not RUN_LIVE_BOOK_LOAN:
    print(
        "Live automation was skipped. Set RUN_LIVE_BOOK_LOAN = True in this cell "
        "only when you intend to allow a real eBiblio loan or reservation attempt."
    )
else:
    print(
        "Starting live eBiblio automation. This may complete a loan or place a reservation."
    )
    live_result = subprocess.run(
        [
            sys.executable,
            "-m",
            "robot",
            "--outputdir",
            str(WORK_DIR / "results"),
            "--variable",
            "HEADLESS:True",
            str(WORK_DIR / "book-loan.robot"),
        ],
        cwd=WORK_DIR,
        check=False,
    )
    if live_result.returncode != 0:
        raise RuntimeError(
            f"Robot Framework live run failed with exit code {live_result.returncode}. "
            "Inspect the generated report and log in the private runtime."
        )
    print(f"Run artifacts were written to {WORK_DIR / 'results'}.")

## 7. View private run reports

After a live run, these links open the Robot Framework report and log in the Colab runtime. They may contain account or session details; do not share them or save them in a shared notebook.

In [ ]:
from IPython.display import FileLink, display

results_dir = WORK_DIR / "results"
report_files = [results_dir / name for name in ("report.html", "log.html")]
available_reports = [path for path in report_files if path.is_file()]
if available_reports:
    for path in available_reports:
        display(FileLink(str(path)))
else:
    print("No live-run reports yet. Run the opt-in live cell first.")